```{contents}
```

## Data Augmentation  (Vision & NLP)


Deep learning models are **data-hungry** and highly sensitive to **data distribution**.
Data augmentation artificially expands the training set by applying label-preserving transformations, improving:

* **Generalization**
* **Robustness to noise and domain shift**
* **Sample efficiency**
* **Regularization** (acts similarly to dropout)

From a learning theory perspective, augmentation **injects invariances** into the model:

* Vision: translation, rotation, illumination invariance
* NLP: lexical, syntactic, and semantic robustness

---

### Formal View

Let original dataset be:
$$
\mathcal{D} = {(x_i, y_i)}_{i=1}^N
$$

Augmentation samples from a transformation family:
$$
\tilde{x} \sim \mathcal{T}(x)
$$

Training objective becomes:
$$
\min_\theta \mathbb{E}*{(x,y)\sim\mathcal{D}} ; \mathbb{E}*{\tilde{x}\sim\mathcal{T}(x)} ; \mathcal{L}(f_\theta(\tilde{x}), y)
$$

Thus the model learns **equivalence classes** of inputs.

---

### Vision Augmentation

#### Common Transformations

| Category    | Examples                           | Effect                 |
| ----------- | ---------------------------------- | ---------------------- |
| Geometric   | Flip, rotate, crop, scale          | Viewpoint invariance   |
| Photometric | Color jitter, brightness, contrast | Lighting robustness    |
| Noise-based | Gaussian noise, blur               | Sensor noise tolerance |
| Structural  | Cutout, MixUp, CutMix              | Strong regularization  |

---

### Vision Workflow

```
Raw Image → Random Transform → Tensor → Model → Loss → Backprop
```

Augmentation is applied **on-the-fly during training**.

---

### PyTorch Demonstration (Vision)

```python
import torchvision.transforms as T

train_aug = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.ToTensor(),
])

from torchvision.datasets import CIFAR10

dataset = CIFAR10(root="./data", train=True, transform=train_aug, download=True)
```

---

### NLP Augmentation

Text cannot be arbitrarily transformed because meaning must be preserved.

#### Common NLP Techniques

| Technique                 | Example                | Effect                   |
| ------------------------- | ---------------------- | ------------------------ |
| Synonym replacement       | good → great           | Lexical robustness       |
| Random deletion           | remove words           | Noise tolerance          |
| Back translation          | en→fr→en               | Paraphrasing             |
| Embedding noise           | perturb embeddings     | Representation smoothing |
| Prompt-based paraphrasing | LLM-generated variants | Semantic diversity       |

---

### NLP Workflow

```
Original Text → Augmentation Function → Tokenizer → Model → Loss → Backprop
```

---

### PyTorch Demonstration (NLP)

```python
import random
from nltk.corpus import wordnet

def synonym_replace(sentence, p=0.2):
    words = sentence.split()
    new_words = []
    for w in words:
        if random.random() < p:
            syns = wordnet.synsets(w)
            if syns:
                new_words.append(syns[0].lemmas()[0].name())
            else:
                new_words.append(w)
        else:
            new_words.append(w)
    return " ".join(new_words)
```

---

### Advanced Augmentation Strategies

| Method                   | Domain | Key Idea                      |
| ------------------------ | ------ | ----------------------------- |
| MixUp                    | Vision | Linear combination of samples |
| CutMix                   | Vision | Patch replacement             |
| RandAugment              | Vision | Learn augmentation policy     |
| SpecAugment              | Speech | Masking in spectrogram        |
| EDA                      | NLP    | Simple rule-based text ops    |
| LLM-Augmentation         | NLP    | Model-generated paraphrases   |
| Adversarial Augmentation | Both   | Worst-case perturbations      |

---

### Remediation: When Augmentation Hurts

| Failure Mode                  | Cause                 | Remedy                          |
| ----------------------------- | --------------------- | ------------------------------- |
| Label noise                   | Aggressive transforms | Constrain transforms            |
| Semantic drift                | Poor NLP rules        | Use embedding similarity filter |
| Underfitting                  | Over-augmentation     | Reduce transform strength       |
| Class imbalance amplification | Uneven augmentation   | Class-aware policies            |

---

### Integration with Training Loop

```python
for images, labels in loader:
    images = images.to(device)
    labels = labels.to(device)

    preds = model(images)
    loss = criterion(preds, labels)

    loss.backward()
    optimizer.step()
```

Augmentation happens **before** the model sees the batch.

---

### Key Design Principles

* Encode domain invariances explicitly
* Keep transformations label-preserving
* Increase diversity without destroying semantics
* Tune augmentation strength with validation performance

---

### Summary Table

| Aspect              | Vision                   | NLP                        |
| ------------------- | ------------------------ | -------------------------- |
| Primary invariances | Geometry, lighting       | Lexical, syntactic         |
| Typical ops         | Flip, crop, jitter       | Synonyms, paraphrase       |
| Risk                | Destroy object semantics | Alter sentence meaning     |
| Best practice       | Policy learning          | Semantic similarity checks |

---

### Final Insight

Data augmentation does not merely create more data —
it **reshapes the hypothesis space** by embedding the structure of the world directly into the learning process.
